In [ ]:
import torch
from torch import nn
from d2l import torch as d2l

def corr2d(X,K):
    """计算二维互相关信息

    Args:
        X (_type_): 二维输入
        K (_type_): 核矩阵
    """
    # 核的大小,行数 列数
    h,w = K.shape
    # 输出
    Y = torch.zeros(X.shape[0]-h+1,X.shape[1]-w+1)

    # 赋值
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i,j] = (X[i:i+h,j:j+w]*K).sum()
    return Y

# 验证上述二维互相关运算的输出
X = torch.tensor([[0.0,1.0,2.0],[3.0,4.0,5.0],[6.0,7.0,8.0]])
K = torch.tensor([[0.0,1.0],[2.0,3.0]])
print(f'验证corr2d(X,K)：\n {corr2d(X,K)}')

class Conv2D(nn.Module):
    def __init__(self,kernel_size):
        self.weight = nn.Parameter(torch.rand(kernel_size))
        self.bias = nn.Parameter(torch.zeros(1))

    def forward(self,x):
        return corr2d(x,self.weight)+self.bias
    
X = torch.ones((6,8))
X[:,2:6] = 0
print(f'X:{X}')

K = torch.tensor([[1,-1]])
Y = corr2d(X,K)

print(f'Y = X()*K:\n {Y}')
print(f'Y = X.t()*K \n {corr2d(X.t(),K)}')

验证corr2d(X,K)：
 tensor([[19., 25.],
        [37., 43.]])
X:tensor([[1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.]])
Y:
 tensor([[ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.]])
X.T*K 
 tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]])


In [12]:
# 反向计算，根据输入输出，计算kernel
conv2d = nn.Conv2d(1,1,kernel_size=(1,2),bias=False)
# 输入 输出维度，y.h = x.h-k.h+1 y.w = x.w-k.w+1
X = X.reshape((1,1,6,8))
Y = Y.reshape((1,1,6,7))
print(f'X:\n {X}')
print(f'Y:\n {Y}')

for i in range(20):
    Y_hat = conv2d(X)
    l = (Y_hat-Y)**2
    conv2d.zero_grad()
    l.sum().backward()
    # w_new = w_old - lr*grad
    conv2d.weight.data[:] -= 3e-2 * conv2d.weight.grad # 3e-2是学习率
    if(i+1) % 2 == 0:
        print(f'batch {i+1},loss {l.sum():.3f}')

# 所学的卷积核的权重张量
print(conv2d.weight.data.reshape((1,2)))

X:
 tensor([[[[1., 1., 0., 0., 0., 0., 1., 1.],
          [1., 1., 0., 0., 0., 0., 1., 1.],
          [1., 1., 0., 0., 0., 0., 1., 1.],
          [1., 1., 0., 0., 0., 0., 1., 1.],
          [1., 1., 0., 0., 0., 0., 1., 1.],
          [1., 1., 0., 0., 0., 0., 1., 1.]]]])
Y:
 tensor([[[[ 0.,  1.,  0.,  0.,  0., -1.,  0.],
          [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
          [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
          [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
          [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
          [ 0.,  1.,  0.,  0.,  0., -1.,  0.]]]])
batch 2,loss 8.157
batch 4,loss 1.748
batch 6,loss 0.449
batch 8,loss 0.139
batch 10,loss 0.049
batch 12,loss 0.019
batch 14,loss 0.008
batch 16,loss 0.003
batch 18,loss 0.001
batch 20,loss 0.001
tensor([[ 0.9975, -1.0022]])
